# Notebook 5 — the scalar ε

Level-1 code: after an absorption, re-emit in the same line with probability $1-\epsilon$ (coherent) or in a line drawn from the thermal emissivity with probability $\epsilon$. Identical blue packets in, an emergent line spectrum out, for $\epsilon = 0, 0.2, 0.5, 1$.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results
from rtedu.visualization import save_fig, OI
from rtedu.atom import five_level_atom, H_ERG_S
rng = np.random.default_rng(rtedu.SEEDS["ch05"])
atom = five_level_atom()
T, n_total, t = 4000.0, 30.0, 2.0 * rtedu.DAY        # the book's standard state from here on
r_out = 0.2 * rtedu.C * t
tau = atom.line_list(T, n_total, t); emis = atom.thermal_emissivity(T, n_total)
nm = 1e7 * atom.lam_cm

## The forest we transport through from now on

The five-level atom of chapter 6, at the book's standard state. Its ten lines and their Sobolev depths:

In [ ]:
for k in range(atom.n_lines):
    print(f"line {k}: {atom.upper[k]}->{atom.lower[k]}  {nm[k]:7.0f} nm  tau_S {tau[k]:7.3f}  thermal emissivity {emis[k]:.3f}")

## Level 1: the ε model in eight lines

In [ ]:
def epsilon_redistribute(rng, k, eps, emis_cum):
    if rng.random() < eps:                          # thermalise: forget the line, draw from the emissivity
        return int(np.searchsorted(emis_cum, rng.random()))
    return k                                        # coherent: the same line
emis_cum = np.cumsum(emis)

def sweep_eps(rng, nu_launch, eps, max_int=200):
    """chapter 4's sweep, plus the redistribution at each interaction"""
    r, nu_lab, labels = 0.0, nu_launch, []
    for _ in range(max_int):
        nu_now = nu_lab * (1 - r / (rtedu.C * t)); r_res = rtedu.C * t * (1 - atom.nu / nu_lab)
        ok = (atom.nu < nu_now) & (r_res < r_out)
        hit = None
        for k in np.flatnonzero(ok)[np.argsort(-atom.nu[ok])]:
            if rng.random() < 1 - np.exp(-tau[k]):
                hit = int(k); break
        if hit is None:
            return nu_lab, labels
        j = epsilon_redistribute(rng, hit, eps, emis_cum); labels.append(("thermal" if j != hit or eps == 1.0 else "scatter", hit, j))
        r = r_res[hit]; nu_lab = atom.nu[j] / (1 - r / (rtedu.C * t))
    return nu_lab, labels

In [ ]:
n = 4000
nu_launch = atom.nu[0] * 1.001                      # a packet just blue of the 455 nm line
eps_values = [0.0, 0.2, 0.5, 1.0]
spectra = {}; n_thermal = {}; n_int = {}
for eps in eps_values:
    last = np.full(n, atom.n_lines); nth = 0; nint = 0
    for i in range(n):
        nu, labels = sweep_eps(rng, nu_launch, eps)
        if labels:
            last[i] = labels[-1][2]
        nth += sum(l[0] == "thermal" for l in labels); nint += len(labels)
    spectra[eps] = np.bincount(last, minlength=atom.n_lines + 1) / n
    n_thermal[eps] = nth / n; n_int[eps] = nint / n
    print(f"eps = {eps:.1f}: {n_int[eps]:.2f} interactions per packet, {n_thermal[eps]:.2f} thermal; escaped without interacting {spectra[eps][-1]:.3f}")

## Validation against `rtedu`

`rtedu.redistribution.EpsilonRedistribution` and `rtedu.transport.run` are the same two pieces.

In [ ]:
from rtedu.redistribution import EpsilonRedistribution
from rtedu.transport import run, emergent_by_line
seed = rtedu.SEEDS["ch05"] + 1
for eps in (0.0, 1.0):
    nu_a, last_a, _ = run(np.random.default_rng(seed), nu_launch, 300, atom.nu, tau, r_out, t, EpsilonRedistribution(eps, emis))
    r2 = np.random.default_rng(seed); last_b = np.full(300, atom.n_lines)
    for i in range(300):
        nu, labels = sweep_eps(r2, nu_launch, eps)
        if labels: last_b[i] = labels[-1][2]
    assert np.array_equal(np.where(last_a < 0, atom.n_lines, last_a), last_b)
print("identical on the same random stream")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
x = np.arange(atom.n_lines + 1); w = 0.2
for i, eps in enumerate(eps_values):
    axes[0].bar(x + (i - 1.5) * w, spectra[eps], w, label=f"eps = {eps}")
axes[0].set_xticks(x); axes[0].set_xticklabels([f"{v:.0f}" for v in nm] + ["none"], rotation=60, fontsize=7); axes[0].set_xlabel("line last emitted in [nm]"); axes[0].set_ylabel("fraction of packets"); axes[0].legend(fontsize=8)
axes[0].set_title("emergent line spectrum of identical blue packets", fontsize=9)
axes[1].plot(eps_values, [n_thermal[e] for e in eps_values], "o-", color=OI["red"], label="thermal re-emissions per packet")
axes[1].plot(eps_values, [n_int[e] for e in eps_values], "s-", color=OI["blue"], label="interactions per packet"); axes[1].set_xlabel(r"$\epsilon$"); axes[1].legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "ch05_epsilon")

In [ ]:
results.record("ch05", dict(T=T, n_total=n_total, t_days=t / rtedu.DAY, v_max_c=0.2, n_packets=n, launch_nm=1e7 * rtedu.C / nu_launch,
                            lines_nm=nm, tau=tau, emis=emis, eps_values=eps_values,
                            spectra={str(e): spectra[e] for e in eps_values}, n_thermal={str(e): n_thermal[e] for e in eps_values},
                            n_int={str(e): n_int[e] for e in eps_values},
                            red_fraction={str(e): float(spectra[e][:atom.n_lines][nm > 1000].sum()) for e in eps_values}))